# Multisurvey filtering with Babamul

**ZTF Summer School · alert streams & filtering session**

In the lecture we said a broker's whole job is to shrink the flood: ten million
detections a night in, a handful of candidates out. In this notebook *you* are the
scientist at the far end of that pipe.

You'll subscribe to live Babamul streams, write filters as plain Python functions,
and — the part that makes this year different — **filter on two surveys at once**,
using ZTF's seven-year baseline to sharpen a Rubin detection and vice versa.

### The one idea to hold on to

Babamul does the expensive work before you ever see an alert: it has already
cross-matched it against other catalogs, scored it with ML, joined it to its
counterpart in the other survey, and fitted its light curve. **Your filter is a
Python function that says yes or no to an object that already knows a great deal
about itself.**

| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | Meet the stream |
| 2 | A filter is a function |
| 3 | Context from cross-matches |
| 4 | **Go multisurvey** |
| 5 | Scan what survived |

This notebook follows the structure of the official
[`stream-basic` example](https://github.com/boom-astro/babamul/tree/main/examples/stream-basic). Once you've done this, that example and its siblings will read easily.

---
## 0 · Setup 

Alongside this notebook you should have a file called **`.env.example`**. Copy it to
`.env` and fill in your credentials:

```bash
cp .env.example .env
```

🚨 **Get your credentials from the `/profile` page of the Babamul deployment you're
using**, and fill in all three values before running anything below. 🚨

| variable | what it's for |
|---|---|
| `BABAMUL_KAFKA_USERNAME` | reading the alert streams |
| `BABAMUL_KAFKA_PASSWORD` | same |
| `BABAMUL_API_TOKEN` | cross-matches, photometry, object lookups, cone searches |

Plus one that picks **which deployment** you're talking to:

| `BABAMUL_ENV` | Kafka | API |
|---|---|---|
| `production` (default) | `kaboom.caltech.edu` | `babamul.caltech.edu` |

In [ ]:
# If you're not using the provided environment:
# %pip install babamul python-dotenv jupyter tqdm astropy ipywidgets matplotlib

In [ ]:
import dotenv
import matplotlib.pyplot as plt
from astropy.table import Table
from collections import Counter
from tqdm.notebook import tqdm

import babamul
import babamul.jupyter
from babamul import LsstAlert, ZtfAlert

In [ ]:
dotenv.load_dotenv()

`load_dotenv()` returns `True` if it found and read a `.env` file. **If it printed
`False`, stop here** — nothing below will authenticate. Check that `.env` sits in the
same directory as this notebook.

The cell below confirms your credentials actually work, and tells you which deployment
you're pointed at. Run it before going further: it turns two common setup mistakes into
a clear message now, rather than a confusing Kafka timeout in Part 1.

In [ ]:
import os

from babamul.config import API_URLS, MAIN_KAFKA_SERVER

required = ["BABAMUL_KAFKA_USERNAME", "BABAMUL_KAFKA_PASSWORD", "BABAMUL_API_TOKEN"]
missing = [key for key in required if not os.environ.get(key)]

if missing:
    print("Missing from your .env:", ", ".join(missing))
    print("Did you copy .env.example to .env and fill it in?")
else:
    env = os.environ.get("BABAMUL_ENV", "production")
    print(f"deployment : {env}")
    print(f"kafka      : {MAIN_KAFKA_SERVER}")
    print(f"api        : {API_URLS[env]}")

    # This is the real test: it round-trips your API token against the server.
    profile = babamul.get_profile()
    print(f"signed in  : {profile.username}")

If `get_profile()` raised an authentication error, your `BABAMUL_API_TOKEN` is wrong or
belongs to the other deployment. Regenerate it from the `/profile` page listed as `api`
in the output above.

---
## 1 · Meet the stream 

Babamul doesn't hand you one undifferentiated firehose. Alerts are pre-sorted into
topics whose names *are* a classification:

```
babamul.{survey}.{match}.{class}
```

* **survey**: `ztf` or `lsst`, whichever produced this alert
* **match**: whether the object has a counterpart in the *other* survey
  (`ztf-match` / `no-ztf-match`, `lsst-match` / `no-lsst-match`)
* **class**: `stellar`, `hosted`, `hostless`, or `unknown`, from catalog matches

So you subscribe to a scientific question, not a stream. Before you write a single
line of filtering logic, **choosing the right topic has already done a lot of the
work**, and the `match` component is where this year's theme lives.

In [ ]:
for t in babamul.topics.ALL_TOPICS:
    print(t)

We'll start where the multisurvey story is: **ZTF alerts, on a galaxy, that LSST has
also seen.** Grab a single alert and look at it.

In [ ]:
topics = ["babamul.ztf.lsst-match.hosted"]

a=None
# Note: spinning up a consumer takes a few seconds. Once alerts flow, it's fast.
with babamul.AlertConsumer(
    topics=topics,
    offset="earliest",
    auto_commit=False,
    timeout=45,
) as consumer:
    for alert in consumer:
        a=alert
        a.show()
        break


`show()` renders the light curve, the cutouts, and, because this object has a
counterpart, the **other survey's photometry on the same axes**.

Now look at what the alert already knows about itself. None of this is something you
had to compute:

In [ ]:
counterpart = a.survey_matches.ztf if a.survey == "LSST" else a.survey_matches.lsst

print(f"object          : {a.objectId}  ({a.survey})")
print(f"real/bogus      : {a.drb}")
print(f"properties      : rock={a.properties.rock}  star={a.properties.star}  "
      f"near_brightstar={a.properties.near_brightstar}  "
      f"stationary={a.properties.stationary}")
print(f"counterpart     : {counterpart.objectId if counterpart else None}")
print(f"detections here : {len(a.get_photometry())}")

**Your turn.** Look at the fitted light-curve statistics Babamul attached.
`properties.photstats` holds per-band results — `peak_jd`, `peak_mag`, `dt`, and
`rising` / `fading` rate fits. These are the fitted parameters from the lecture,
handed to you as filterable fields.

In [ ]:
# TODO: print the per-band fitted stats for this alert.
#       For each band that isn't None, show peak_mag and the rising/fading rates.
stats = a.properties.photstats

for band in ["u", "g", "r", "i", "z", "y"]:
    bp = ...
    if bp is None:
        continue
    ...

In [ ]:
# SOLUTION
stats = a.properties.photstats

for band in ["u", "g", "r", "i", "z", "y"]:
    bp = getattr(stats, band, None)
    if bp is None:
        continue
    rise = f"{bp.rising.rate:+.3f} mag/d" if bp.rising else "   --    "
    fade = f"{bp.fading.rate:+.3f} mag/d" if bp.fading else "   --    "
    peak = f"{bp.peak_mag:.2f}" if bp.peak_mag is not None else " -- "
    print(f"{band}: peak {peak}  dt={bp.dt:6.1f} d   rising {rise}   fading {fade}")

---
## 2 · A filter is a function 

Here's the shift from how you might have imagined this working. A Babamul filter is
**an ordinary Python predicate**: take an alert, return `True` or `False`. No query
language, no DSL.

That means you get the whole language: intermediate variables, loops over photometry,
`and`/`or` however you like. It also means cheap cuts should come first, because
you're running this over a live stream.

Below is a starting filter, adapted from the official example. Read it before running
it, every condition is a decision someone made, and you're about to change them.

In [ ]:
def is_relevant(alert: ZtfAlert | LsstAlert) -> bool:
    """Cheap, per-alert cuts. Order matters: fastest and most selective first."""

    # --- real/bogus: the highest-value single cut in alert filtering
    if alert.drb is not None and alert.drb < 0.4:
        return False

    # --- brightening, not fading away
    if not alert.candidate.isdiffpos:
        return False

    # --- pre-computed vetoes, free at this point
    if alert.properties.rock:            # known solar system object
        return False
    if alert.properties.star:            # PS1 PSC for ZTF, LSPSC for LSST
        return False
    if alert.properties.near_brightstar:
        return False

    # --- survey-specific quality
    if isinstance(alert, ZtfAlert):
        age = alert.candidate.jd - alert.candidate.jdstarthist
        if age < 3 or age > 60:          # young, but not brand new
            return False

    return True

Run it over the stream and collect what passes.

In [ ]:
alerts: list[ZtfAlert | LsstAlert] = []
LIMIT = 500

with babamul.AlertConsumer(
    topics=topics, offset="earliest", auto_commit=False, timeout=15,
) as consumer:
    for alert in tqdm(consumer, desc="filtering"):
        if is_relevant(alert):
            alerts.append(alert)
        if len(alerts) >= LIMIT:
            break

print(f"kept {len(alerts)} alerts")

One object can produce many alerts. Collapse to the most recent alert per object, 
otherwise you'll fetch the same cross-matches over and over in the next section.

In [ ]:
by_object = {}
for a in alerts:
    if a.objectId not in by_object or a.candidate.jd > by_object[a.objectId].candidate.jd:
        by_object[a.objectId] = a

to_scan = list(by_object.values())
print(f"{len(alerts)} alerts -> {len(to_scan)} unique objects")

**Your turn.** `drb < 0.4` is loose. The example gets away with it because Babamul's
topics have already removed a lot of junk. Find out what it's actually costing you.

In [ ]:
# TODO: sweep the drb threshold. For each value, count how many of the alerts
#       you already collected would pass.
#       (Filter the `alerts` list you have — no need to re-consume the stream.)

for thr in [...]:
    n = ...
    print(f"drb >= {thr}: {n} alerts")

In [ ]:
# SOLUTION
for thr in [0.2, 0.4, 0.6, 0.8, 0.95]:
    n = sum(1 for a in alerts if a.drb is not None and a.drb >= thr)
    print(f"drb >= {thr:<5} {n:>4} alerts  ({100*n/len(alerts):.1f}% of what we kept)")

# Because the topic already excluded stellar and rock-like alerts, drb is doing
# less work here than it would on the raw survey stream. That's the point of
# pre-sorted topics: the cut you'd reach for first has partly been made for you.

---
## 3 · Context from cross-matches

The cuts so far ask *is this detection real?* They say nothing about *what it is*.
For that you need to know what was already at those coordinates.

`add_cross_matches` fetches them in bulk from the API and attaches them to each alert.

In [ ]:
babamul.add_cross_matches(to_scan, n_threads=8)

xm = to_scan[0].cross_matches
print("catalogs available on an alert:")
for name in ["ned", "gaia", "lspsc", "milliquasar", "catwise", "vsx"]:
    matches = getattr(xm, name) or []
    print(f"  {name:<12} {len(matches)} match(es)")

Each catalog answers a different question:

| catalog | what it tells you |
|---|---|
| `ned` | is there a known galaxy here, and at what redshift? |
| `gaia` | does this source have a parallax or proper motion? → it's a star |
| `lspsc` | Legacy Survey point sources, with a star/galaxy `score` |
| `milliquasar` | is this a known quasar? → the AGN veto |
| `vsx` | is this a known variable star? |

**Your turn.** Write the context filter. You want objects that sit *near* a galaxy but
not *on* its nucleus, that aren't stellar, and that aren't a known AGN.

In [ ]:
# TODO: implement the cross-match filter.
#
#   keep an object if:
#     - it has at least one NED match (a known galaxy is nearby), AND
#     - no NED match is within 2 arcsec (that would be nuclear, not a SN), AND
#     - it has no Gaia match (Gaia sources are stars), AND
#     - it has no milliquasar match (known AGN)
#
#   hint: `m.distance_arcsec` on each match

def has_good_context(alert: ZtfAlert | LsstAlert) -> bool:
    xm = alert.cross_matches
    ...


with_context = [a for a in to_scan if has_good_context(a)]
print(f"{len(with_context)} of {len(to_scan)} objects survive the context cuts")

In [ ]:
# SOLUTION
def has_good_context(alert: ZtfAlert | LsstAlert) -> bool:
    xm = alert.cross_matches

    ned = xm.ned or []
    if not ned:
        return False                                    # no known host nearby
    if any(m.distance_arcsec is not None and m.distance_arcsec < 2.0 for m in ned):
        return False                                    # nuclear -> likely AGN/TDE
    if xm.gaia:
        return False                                    # Gaia sees it -> stellar
    if xm.milliquasar:
        return False                                    # known quasar
    return True


with_context = [a for a in to_scan if has_good_context(a)]
print(f"{len(with_context)} of {len(to_scan)} objects survive the context cuts")

Note the shape of that. The *nuclear* cut is the interesting one: rejecting anything
within 2 arcsec of a galaxy centre removes AGN variability, but it also removes
genuine tidal disruption events, which are nuclear by definition. If TDEs are your
science, you'd invert exactly that condition.

---
## 4 · Go multisurvey

**This is the part that matters this year.**

You already made one multisurvey decision without writing any code: you subscribed to
`babamul.ztf.lsst-match.hosted` rather than `no-lsst-match`. Every alert in this
notebook has a counterpart by construction.

### 4a · The counterpart is already attached

`survey_matches` carries the other survey's photometry on the alert itself, no second
query.

In [ ]:
a = (with_context or to_scan)[0]
match = a.survey_matches.ztf if a.survey == "LSST" else a.survey_matches.lsst

print(f"this alert  : {a.objectId}  ({a.survey})")
print(f"counterpart : {match.objectId}")
print(f"  detections here        : {len(a.get_photometry())}")
print(f"  detections there       : {len(match.prv_candidates)}")
print(f"  forced photometry there: {len(match.fp_hists)}")

Both surveys' photometry arrives already converted to a common magnitude scale, so you
can put them straight on one axis, no zero-point juggling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

here = [p for p in a.get_photometry() if p.magpsf is not None]
there = [p for p in match.prv_candidates if p.magpsf is not None]

ax.errorbar([p.jd for p in here], [p.magpsf for p in here],
            yerr=[p.sigmapsf or 0 for p in here],
            fmt='o', alpha=0.8, label=f"{a.survey} {a.objectId}")
ax.errorbar([p.jd for p in there], [p.magpsf for p in there],
            yerr=[p.sigmapsf or 0 for p in there],
            fmt='s', alpha=0.8, label=f"counterpart {match.objectId}")

ax.invert_yaxis()
ax.set_xlabel("JD")
ax.set_ylabel("magnitude")
ax.set_title("One object, two surveys")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4b · The field that makes this easy

Babamul fits the light curve twice: once on this survey alone
(`properties.photstats`) and once on the **merged, cross-survey** curve
(`properties.multisurvey_photstats`). Same structure, different input data.

That second field is this year's theme in one attribute. Compare them.

In [ ]:
# TODO: for one alert, print the fitted rise rate per band from photstats
#       and from multisurvey_photstats side by side.
#
#       Where do they disagree, and why would the merged fit be better?

single = a.properties.photstats
merged = a.properties.multisurvey_photstats

for band in ["u", "g", "r", "i", "z", "y"]:
    ...

In [ ]:
# SOLUTION
single = a.properties.photstats
merged = a.properties.multisurvey_photstats


def describe(bp):
    if bp is None:
        return "        --         "
    r = f"{bp.rising.rate:+.3f}" if bp.rising else "  --  "
    n = bp.rising.nb_data if bp.rising else 0
    return f"rise {r} ({n:>3} pts)"


print(f"{'band':<6}{'single-survey':<24}{'multisurvey':<24}")
for band in ["u", "g", "r", "i", "z", "y"]:
    s = getattr(single, band, None)
    m = getattr(merged, band, None) if merged else None
    if s is None and m is None:
        continue
    print(f"{band:<6}{describe(s):<24}{describe(m):<24}")

# The merged fit usually has more points and a longer baseline, so its rate has a
# smaller uncertainty — and in bands one survey barely samples, it may be the only
# fit that exists at all.

### 4c · A filter only a multisurvey broker can run

Now write something genuinely impossible with one survey. Three ideas:

1. **Rise confirmed across surveys**: require a rising fit in
   `multisurvey_photstats` built from more points than the single-survey fit had.
2. **Quiet history, loud now**: the counterpart's photometry goes back years. No
   previous detections plus bright tonight means a real new transient, not a variable
   caught mid-cycle.
3. **Not a repeat offender**: if the counterpart's history shows several separate
   brightening episodes, it's a CV or an AGN, whatever tonight's alert looks like.

Implement 2 or 3, whichever you find more interesting.

In [ ]:
# TODO: write a cross-survey filter.
#
# Option A: "quiet history, loud now":
#   count detections in the counterpart's prv_candidates more than 100 days
#   before this alert's jd. Keep objects where that count is 0.
#
# Option B: "not a repeat offender":
#   walk the counterpart's photometry in time order and count how many separate
#   times it brightens by more than 1 mag and fades again. Reject if > 1.

def passes_history_check(alert) -> bool:
    match = alert.survey_matches.ztf if alert.survey == "LSST" else alert.survey_matches.lsst
    if match is None:
        return False
    ...


final = [a for a in with_context if passes_history_check(a)]
print(f"{len(final)} objects pass the cross-survey history check")

In [ ]:
# SOLUTION: Option A: quiet history, loud now
def passes_history_check(alert) -> bool:
    match = alert.survey_matches.ztf if alert.survey == "LSST" else alert.survey_matches.lsst
    if match is None:
        return False

    now = alert.candidate.jd
    old_detections = [
        p for p in match.prv_candidates
        if p.magpsf is not None and p.isdiffpos and p.jd < now - 100
    ]
    return len(old_detections) == 0


final = [a for a in with_context if passes_history_check(a)]
print(f"{len(final)} objects pass the cross-survey history check")
print(f"  (from {len(with_context)} after context cuts, "
      f"{len(to_scan)} unique objects, {len(alerts)} raw alerts)")

# Worth noticing: this rejects objects that LOOK new in tonight's survey but have
# years of history in the other one. No single-survey filter can see that.

Look at the whole funnel you just built.

In [ ]:
stages = [
    ("raw alerts consumed", len(alerts)),
    ("unique objects", len(to_scan)),
    ("+ cross-match context", len(with_context)),
    ("+ cross-survey history", len(final)),
]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(range(len(stages)), [s[1] for s in stages], color="#2A9D8F")
ax.set_yticks(range(len(stages)))
ax.set_yticklabels([s[0] for s in stages], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("surviving")
for i, (_, n) in enumerate(stages):
    ax.text(n, i, f" {n}", va="center", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5 · Scan what survived 

This is the moment a real scanner reaches: a short list, and eyes on each one.
`scan_alerts` gives you a pager: light curve, cutouts, and the counterpart's
photometry overlaid.

In [ ]:
babamul.jupyter.scan_alerts(
    sorted(final or with_context, key=lambda a: a.candidate.magpsf or 99),
    include_survey_matches=True,
    include_nondetections=True,
)

Page through them. For each, ask the question the filter couldn't: **would you spend
telescope time on this tonight?**

If more than one or two look obviously wrong, that's information. Go back and work out
which condition should have caught them.

---
## Wrap-up

You just did, by hand, what Babamul does continuously: **pick the right stream, apply
cheap cuts first, add cross-match context, join across surveys, eyeball what survives,
and check you didn't cast too wide a net.**

Three things worth carrying away:

1. **Choosing the topic is filtering.** `babamul.lsst.ztf-match.hosted` made three
   decisions before your code ran.
2. **Every cut is a trade.** No threshold removes only junk. Know which direction you'd
   rather be wrong in.
3. **`multisurvey_photstats` is the whole point.** A light curve fitted across two
   surveys has a longer baseline and more bands than either alone, and it's a field
   you filter on, not a project you have to run.
4. **Measure your recovery rate.** "What does my filter return?" is a much weaker
   question than "which known objects would it have missed, and why?"

### Where to go next

* [`examples/stream-cached`](https://github.com/boom-astro/babamul/tree/main/examples/stream-cached)
  : keeping state across runs so you don't reprocess alerts
* [`examples/stream-app`](https://github.com/boom-astro/babamul/tree/main/examples/stream-app)
  : the same logic as a long-running service instead of a notebook
* [`examples/api`](https://github.com/boom-astro/babamul/tree/main/examples/api)
  : archival searches, including cone searches against a galaxy catalog

*Structure and filtering patterns adapted from the official `boom-astro/babamul`
`stream-basic` example.*